# Data Analysis

Answer the selected question, “Which delivery time slots cause the most late orders?”, using approval time as the time-slot anchor. Delivery outcomes count distinct orders.

**Contents**

1. [Project setup and input](#analysis-setup) — Load the cleaned item table.
2. [Order-level delivery outcomes](#analysis-orders) — Classify orders, assign approval slots and calculate lateness.
3. [Late delivery by approval slot](#analysis-slot-summary) — Compare order counts, late rates and delay duration.
4. [Delivery patterns in figures](#analysis-figures) — Inspect four slots, hourly and weekday rates, and delay duration.
5. [Statistical comparison of slots](#analysis-statistics) — Test equal late rates and assess the size of the difference.
6. [Temporal and cohort stability](#analysis-stability) — Compare months, matched years and cleaning cohorts.
7. [Delivery stages and promised dates](#analysis-stages) — Compare elapsed time, eligible-day time and delivery promises.
8. [Route composition and delivery](#analysis-routes) — Compare route types, approval slots and carrier transit.

The slot test uses the summary in section 3, the cleaning comparison also reads the full merged table, and the route comparison uses the stage durations from section 7.

[Cleaning User Guide](../docs/Cleaning/User_Guide.md) · [Cleaning Technical Reference](../docs/Cleaning/Technical_Reference.md)


<a id="analysis-setup" name="analysis-setup"></a>

## 1. Project setup and input

Locate the project and load the cleaned order-item table.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    # Resolve imports from either the repository root or notebooks directory.
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
import matplotlib.pyplot as plt

from src.analysis.presentation import format_analysis_table
from src.analysis.orders import build_order_analysis_view
from src.analysis.summary import summarize_delivery_by_slot
from src.analysis.statistics import test_slot_association

from src.analysis.plots import (
    build_temporal_profiles, plot_slot_counts, plot_rate_profile,
    plot_hour_weekday_heatmap, plot_delay_distribution,
)
from src.data_loader import load_data
from src.paths import PROCESSED_DIR

from src.analysis.investigation import (
    summarize_time_periods, compare_cleaning_cohorts, investigate_delivery_stages,
    classify_order_routes, summarize_routes,
)
from src.analysis.investigation_plots import (
    plot_monthly_slot_rates, plot_matched_years, plot_stage_comparison, plot_promised_windows,
    plot_route_slot_rates, plot_route_transit_time,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed input: {PROCESSED_DIR}")


### Load the cleaned item table


In [ ]:
cleaned_items = load_data("cleaned_order_items", stage="processed")


<a id="analysis-orders" name="analysis-orders"></a>

## 2. Order-level delivery outcomes

Keep each order once in the analysis view. Orders with missing or conflicting order-level values retain all their item rows in the review view. One reviewed order may have several reasons. Approval time determines one of four six-hour slots. A delivery is late only when its calendar date is after the estimated date.


In [ ]:
orders, order_diagnostics = build_order_analysis_view(cleaned_items)
display(order_diagnostics["cohort_summary"])

if not order_diagnostics["review_reasons"].empty:
    print("Orders requiring review by field and issue")
    display(order_diagnostics["review_reasons"].groupby(["field", "issue"]).size().rename("orders").reset_index())
    print("First 20 review reasons; complete records remain in order_diagnostics['review_reasons'].")
    display(order_diagnostics["review_reasons"].head(20))
    print("Original item rows remain in order_diagnostics['review_items'].")

print("First five analyzable orders")
display(orders[[
    "order_id", "order_approved_at", "approval_slot",
    "order_estimated_delivery_date", "order_delivered_customer_date",
    "is_late", "delay_days",
]].head())


<a id="analysis-slot-summary" name="analysis-slot-summary"></a>

## 3. Late delivery by approval slot

Compare the four approval-time slots at order grain. Late rate divides late orders by all analyzable orders in the slot. Mean and median delay use late orders only.


In [ ]:
slot_summary = summarize_delivery_by_slot(orders)
display(format_analysis_table(slot_summary))

if orders.empty:
    print("No analyzable orders are available for slot comparisons.")
elif not orders["is_late"].any():
    print("No late orders are available for a late-delivery ranking.")
else:
    for label, measure in [
        ("Most late orders", "late_orders"),
        ("Highest late rate", "late_rate_pct"),
        ("Longest mean delay among late orders", "mean_delay_days"),
    ]:
        maximum = slot_summary[measure].max()
        leaders = slot_summary.loc[slot_summary[measure].eq(maximum), "approval_slot"]
        print(f"{label}: {', '.join(leaders)}")


For the current cleaned dataset, 12:00–18:00 has the most late orders (2,085) and the largest order volume. The highest late rate is 00:00–06:00 (7.31%). Among late orders, 06:00–12:00 has the longest mean delay (11.18 days), while every slot has a median of seven days. These rankings describe different aspects of delivery performance.


<a id="analysis-figures" name="analysis-figures"></a>

## 4. Delivery patterns in figures

Use the same order-level data as the summary table. Each rate uses orders in its own group as the denominator. Hour and weekday profiles include every recorded hour and weekday.


In [ ]:
temporal_profiles = build_temporal_profiles(orders)


### On-time and late deliveries by approval slot

Each stack counts all delivered orders approved in one slot. The labels give the total and the number delivered late.


In [ ]:
slot_counts_figure = plot_slot_counts(slot_summary)
display(slot_counts_figure)
plt.close(slot_counts_figure)


The 12:00–18:00 slot has both the largest order volume and the most late orders. Its late rate remains close to the overall rate, as shown in the summary table.


### Late delivery rate by approval hour

Each rate uses the orders approved in that hour of the recorded clock. N gives the number of orders.


In [ ]:
hourly_rate_figure = plot_rate_profile(temporal_profiles["hours"], "approval_hour")
display(hourly_rate_figure)
plt.close(hourly_rate_figure)


Among individual hours, 03:00–03:59 has the highest observed late rate (7.85%, 422 of 5,378 orders). The busiest approval hour is 02:00–02:59, with 6,855 orders and a 7.66% late rate.


### Late delivery rate by approval weekday

Each rate uses all orders approved on that weekday, across all hours. N gives the number of orders.


In [ ]:
weekday_rate_figure = plot_rate_profile(temporal_profiles["weekdays"], "approval_weekday")
display(weekday_rate_figure)
plt.close(weekday_rate_figure)


Across all approval hours, Friday has the highest observed weekday late rate (7.43%, 1,049 of 14,119 orders).


### Late delivery rates and order counts by approval hour and weekday

Darker green indicates a higher share of late deliveries. Darker blue indicates more orders. Grey marks an hour and weekday with no orders. The table lists the ten highest observed rates with their order counts.


In [ ]:
hour_weekday_figure = plot_hour_weekday_heatmap(temporal_profiles["hour_weekday"])
display(hour_weekday_figure)
plt.close(hour_weekday_figure)
display(
    temporal_profiles["hour_weekday"].sort_values("late_rate_pct", ascending=False)
    .head(10).rename(columns={
        "approval_weekday": "Weekday (Mon=0)",
        "approval_hour": "Approval hour",
        "orders": "Orders",
        "late_orders": "Late orders",
        "late_rate_pct": "Late rate (%)",
    }).style.format({"Late rate (%)": "{:.2f}", "Orders": "{:,.0f}", "Late orders": "{:,.0f}"})
    .hide(axis="index")
)


The highest observed hour-by-weekday rate is Sunday 00:00–00:59, at 12.66% (30 of 237 orders). The paired count panel shows why a high-rate cell should be read alongside its sample size.


### Days past the promised date among late orders

The boxplots show calendar days late by approval slot. One view covers the full distribution. The detailed view ends at the 99th percentile across all late orders. Both use the full samples to calculate the boxes, medians and means.


In [ ]:
delay_distribution_figure = plot_delay_distribution(orders)
display(delay_distribution_figure)
plt.close(delay_distribution_figure)


Every approval slot has a median delay of seven days among late orders. The means differ more, so the upper tail contributes to the severity ranking.


<a id="analysis-statistics" name="analysis-statistics"></a>

## 5. Statistical comparison of slots

The Chi-square test evaluates whether late-delivery rates differ across the four approval slots. The null hypothesis is that all four rates are equal. The alternative is that at least one rate differs. The significance level is 5%.


### Observed delivery outcomes

The contingency table counts on-time and late deliveries in each approval slot. The test derives expected counts from the row and column totals under the assumption of equal late rates.


In [ ]:
slot_test = test_slot_association(slot_summary, alpha=0.05)
display(slot_test["observed"].rename_axis("Approval slot").rename(columns={
    "on_time_orders": "On time", "late_orders": "Late",
}).style.format("{:,.0f}"))
failed_checks = slot_test["checks"].loc[lambda table: ~table["passed"]]
if not failed_checks.empty:
    display(failed_checks[["check"]].rename(columns={"check": "Unmet test condition"}).style.hide(axis="index"))


### Test result and size of the association

The p-value measures how unusual the overall difference would be under equal rates and the test assumptions. Cramer's V describes the strength of association. The percentage-point gap compares the highest and lowest observed rates.


In [ ]:
test_result = slot_test["result"].iloc[0]
test_metrics = slot_test["result"][[
    "orders", "alpha", "chi2", "dof", "p_value",
    "min_expected", "cramers_v", "rate_range_pp",
]].rename(columns={
    "orders": "Analyzable orders",
    "alpha": "Significance threshold",
    "chi2": "Chi-square statistic",
    "dof": "Degrees of freedom",
    "p_value": "P-value",
    "min_expected": "Smallest expected count",
    "cramers_v": "Cramér's V",
    "rate_range_pp": "Highest minus lowest rate (percentage points)",
}).T
test_metrics.columns = ["Value"]
display(test_metrics.style.format("{:,.6g}", na_rep="—"))
print(test_result["decision"])


Late-delivery rates differ across approval slots in the analyzed cohort, with Chi-square = 13.78, 3 degrees of freedom and p = 0.0032. The minimum expected count is 1,233, satisfying the frequency criterion.

The largest observed rate difference is 0.96 percentage points, and Cramer's V is 0.012. This indicates a small association. The large sample provides evidence of a difference even though its absolute size is modest.


### Interpretation limits

The Chi-square p-value applies to the four-slot table as a whole. The slot rankings describe observed rates; the overall test does not establish which individual pairs differ.

Orders sharing an approval date or seller may be exposed to common operating conditions, which can violate the independence assumption. The p-value therefore depends on an assumption that cannot be fully verified from these records.

The findings apply to delivered orders retained after cleaning. Differences in season, geography or promised delivery time may also contribute to the observed association. The results alone do not establish the effect of changing warehouse or carrier schedules.


<a id="analysis-stability" name="analysis-stability"></a>

## 6. Temporal and cohort stability

Compare slot rates across purchase months, matched January-July periods, and before versus after cleaning. Purchase dates select the time cohort; approval times assign the slots.


### Monthly late delivery rates and order counts by approval slot

Orders are grouped by purchase month and colored by approval slot. Rates are shown on the full observed range and a 0–25% scale. The count series gives the denominator for each month and slot. A gap in a rate series indicates zero orders.


In [ ]:
period_profiles = summarize_time_periods(orders)
monthly_counts = period_profiles["monthly"].groupby("month", as_index=False)[["orders", "late_orders"]].sum()
display(format_analysis_table(monthly_counts))


In [ ]:
analysis_monthly_figure = plot_monthly_slot_rates(period_profiles["monthly"])
display(analysis_monthly_figure)
plt.close(analysis_monthly_figure)


September 2016 contains one order and November 2016 has none. August 2018 covers purchases through 29 August and contains 6,261 orders. The matched-year comparison uses January–July in both years.


### Late delivery rates for January–July purchases in 2017 and 2018

Each bar uses orders purchased in January–July of the indicated year. Approval time determines the slot. N gives the number of orders in that group.


In [ ]:
display(format_analysis_table(period_profiles["matched_years"][[
    "year", "approval_slot", "orders", "late_orders", "late_rate_pct",
]]))


In [ ]:
analysis_years_figure = plot_matched_years(period_profiles["matched_years"])
display(analysis_years_figure)
plt.close(analysis_years_figure)


In January-July 2017, 18:00–24:00 has the highest rate (3.88%), closely followed by 00:00–06:00 (3.85%). In the same months of 2018, 00:00–06:00 ranks highest (8.45%). Every slot has a higher observed rate in 2018, so the pooled ranking also reflects the time composition of orders.


### Before and after cleaning

The comparison starts with delivered orders whose approval, customer-delivery and estimated-delivery timestamps are available and consistent within the order. Compare their four-slot rates with the analyzable cleaned cohort. Missing comparison timestamps and conflicting orders are counted separately.


In [ ]:
merged_items = load_data("merged_order_items", stage="interim")
cohort_comparison = compare_cleaning_cohorts(merged_items, orders)
display(format_analysis_table(cohort_comparison["coverage"]))
if cohort_comparison["slots"].empty:
    print("Comparison unavailable. Check the missing or changed orders in the coverage table.")
else:
    display(format_analysis_table(cohort_comparison["slots"][[
        "cohort", "approval_slot", "orders", "late_orders", "late_rate_pct",
    ]]))


For the analyzed Olist extract, 22 of 96,478 delivered orders lack at least one comparison timestamp. The comparison therefore uses 96,456 orders before cleaning and 95,075 after cleaning. The 12:00–18:00 slot has the most late orders and 00:00–06:00 has the highest late rate in both cohorts.


<a id="analysis-stages" name="analysis-stages"></a>

## 7. Delivery stages and promised dates

Measure approval to carrier handoff, then carrier handoff to customer delivery. Recorded elapsed hours preserve the full wait. Eligible-day hours count Monday-Friday except 1 January, 1 May and 25 December, with 24 hours per eligible day and partial endpoint days retained. These hours describe a calendar convention rather than actual staffed hours.


In [ ]:
stage_profiles = investigate_delivery_stages(orders)
display(format_analysis_table(stage_profiles["quality"]))
display(format_analysis_table(stage_profiles["summary"]))
print("Median hours across all approval slots")
display(format_analysis_table(stage_profiles["pooled_medians"]))


### Time from approval to carrier handoff

Bars show the median time to handoff by approval slot for on-time and late deliveries. N counts orders with a valid interval in each group.


In [ ]:
analysis_processing_figure = plot_stage_comparison(stage_profiles["summary"], "Processing")
display(analysis_processing_figure)
plt.close(analysis_processing_figure)


Across all slots, median recorded approval-to-handoff time is 73.60 hours for late orders and 43.01 hours for on-time orders. Under the eligible-day convention, the corresponding medians are 50.15 and 30.10 hours. These retrospective groups differ in the time before handoff.


### Time from carrier handoff to customer delivery

Bars show the median transit time by approval slot and delivery outcome. Elapsed hours cover the full interval. Eligible hours count the part of that interval on eligible days.


In [ ]:
analysis_transit_figure = plot_stage_comparison(stage_profiles["summary"], "Carrier transit")
display(analysis_transit_figure)
plt.close(analysis_transit_figure)


Median recorded carrier-to-customer time is 628.75 hours for late orders and 166.95 hours for on-time orders. Eligible-day medians are 438.05 and 118.22 hours. Late status also depends on the promised date, so these duration differences alone cannot identify the cause of a late delivery.


### Days from approval to the promised delivery date

The interval uses calendar dates. A negative value identifies an order approved after its promised date. The boxplots show the full range by approval slot.


In [ ]:
display(format_analysis_table(stage_profiles["promised_summary"]))
print(f"Orders approved after the estimated delivery date: {len(stage_profiles['negative_windows']):,}")
if not stage_profiles["negative_windows"].empty:
    display(format_analysis_table(stage_profiles["negative_windows"].head(10)))


In [ ]:
analysis_promise_figure = plot_promised_windows(stage_profiles["windows"])
display(analysis_promise_figure)
plt.close(analysis_promise_figure)


Median promised windows are 22, 23, 24 and 24 calendar days for the four slots, respectively. The earliest approval slot has a shorter typical promise window, which should be considered alongside its higher late rate. One negative window is retained in the displayed distribution and review table.


<a id="analysis-routes" name="analysis-routes"></a>

## 8. Route composition and delivery

Classify each analyzed order from all its item-level seller and customer states. Intra-state means every item has matching states; inter-state means every item differs. Mixed orders contain both patterns. Any missing state makes the route unknown. State agreement is a geographic proxy, not a measured distance or actual carrier route.


### Route coverage and approval slots

Count each order once in its route group. The four-slot table retains empty groups with undefined rates. Each late rate divides late orders by all orders in that route and slot.


In [ ]:
order_routes = classify_order_routes(cleaned_items, orders)
route_profiles = summarize_routes(orders, order_routes, stage_profiles["records"])
print("Route totals")
display(format_analysis_table(route_profiles["totals"]))
print("Route groups by approval slot")
display(format_analysis_table(route_profiles["slots"]))


The 60,776 inter-state orders include 4,950 late deliveries (8.14%), compared with 1,558 of 34,059 intra-state orders (4.57%). The observed gap is 3.57 percentage points. The 240 mixed orders include one late delivery; there are no unknown routes in this cohort. The inter-state group accounts for 76.05% of late orders and 63.92% of all orders. These differences describe route composition, without separating it from other factors associated with destinations or sellers.


### Intra-state and inter-state late delivery rates by approval slot

The rate uses all orders in each route and slot. Intra-state orders have matching seller and customer states on every item. Inter-state orders have different states on every item.


In [ ]:
analysis_route_slot_figure = plot_route_slot_rates(route_profiles["slots"])
display(analysis_route_slot_figure)
plt.close(analysis_route_slot_figure)


Within intra-state orders, 00:00–06:00 has the highest observed late rate (5.46%, 393 of 7,198). Within inter-state orders, 00:00–06:00 is also highest (8.40%, 1,062 of 12,649), closely followed by 18:00–24:00 (8.37%, 1,433 of 17,112). The slot ranking is therefore less distinct within inter-state deliveries than the pooled ranking alone suggests.


### Median carrier transit time by route type

The table covers all route types. The figure compares intra-state and inter-state orders using elapsed hours and eligible hours. N counts orders with a valid handoff-to-delivery interval.


In [ ]:
display(format_analysis_table(route_profiles["transit"]))


In [ ]:
analysis_route_transit_figure = plot_route_transit_time(route_profiles["transit"])
display(analysis_route_transit_figure)
plt.close(analysis_route_transit_figure)


Median carrier transit is 87.77 recorded hours for intra-state orders and 221.04 for inter-state orders. Under the eligible-day convention, the medians are 64.55 and 164.25 hours. All 95,075 orders have valid transit intervals. Longer observed transit in the inter-state group is consistent with the geographic proxy, but state equality alone does not measure distance or establish why an individual order was late.
